# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [1]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...


Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Requiem dla snu
2. Wielkowiejski Targ
3. Make Emo a Threat Again Festival
4. Pewnego długiego dnia
5. Britney Manson w Zaścianku
6. Punky Reggae live w Studio
7. Solvay Nocą: Noce i nihilify
8. The Strings 2 jadą do USA
9. Marzyciele
10. Komeda: Perspektywa Nova w Forcie Borek

Czas wykonania: 2.97s


In [2]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...


Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Requiem dla snu
2. Wielkowiejski Targ
3. Make Emo a Threat Again Festival
4. Pewnego długiego dnia
5. Britney Manson w Zaścianku
6. Punky Reggae live w Studio
7. Solvay Nocą: Noce i nihilify
8. The Strings 2 jadą do USA
9. Marzyciele
10. Komeda: Perspektywa Nova w Forcie Borek

Czas wykonania (wątki): 0.69s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [3]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [4]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...


Zakończono w czasie 1.09s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [5]:
import requests
import time
import concurrent.futures

url = "https://catfact.ninja/fact"

start = time.time()
for _ in range(20):
    requests.get(url)
print(f"sekw: {time.time() - start} s")

start2 = time.time()
with concurrent.futures.ThreadPoolExecutor() as ex:
    ex.map(lambda _: requests.get(url), range(20))
print(f"watki: {time.time() - start2} s")

sekw: 3.240504741668701 s


watki: 0.3973057270050049 s


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [6]:
import queue
import threading
import time

q = queue.Queue()

def p():
    for i in range(20):
        q.put(i)
        time.sleep(0.01)

def c1():
    while True:
        try:
            x = q.get(timeout=1)
            if x % 2 == 0: print(f"p: {x}")
            else: q.put(x)
        except: break

def c2():
    while True:
        try:
            x = q.get(timeout=1)
            if x % 2 != 0: print(f"np: {x}")
            else: q.put(x)
        except: break

t1 = threading.Thread(target=p)
t2 = threading.Thread(target=c1)
t3 = threading.Thread(target=c2)

t1.start(); t2.start(); t3.start()
t1.join(); t2.join(); t3.join()

p: 0
np: 1
p: 2
np: 3
p: 4
np: 5
p: 6
np: 7
p: 8
np: 9
p: 10
np: 11
p: 12
np: 13
p: 14
np: 15
p: 16
np: 17


p: 18
np: 19


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [7]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    t = time.time()
    with multiprocessing.Pool() as p:
        p.map(calculate_power_sum, range(1, 1000))
    print(time.time() - t)

0.2619304656982422
